<a href="https://colab.research.google.com/github/NJerez-dev/Analisis-Datos-Transporte-UM/blob/main/02_silver_transformacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥈 Notebook 02 — Capa SILVER
**Propósito:** Limpiar, tipar, normalizar y enriquecer los datos Bronze.  
**Principio Silver:** Datos confiables, tipados correctamente, sin duplicados, listos para analytics.

| Tabla Bronze | → | Tabla Silver | Transformaciones |
|---|---|---|---|
| `bronze_viajes_diarios` | → | `silver_viajes` | Tipos, nulos, Volumen numérico, tiempos |
| `bronze_devoluciones` | → | `silver_devoluciones` | Fechas parseadas, estados limpios |

## 0️⃣ Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import sqlite3
import numpy as np
from datetime import datetime

DRIVE_BASE = '/content/drive/MyDrive/SUPPLY_CHAIN_PROJECT'
DB_PATH    = f'{DRIVE_BASE}/supply_chain.db'

con = sqlite3.connect(DB_PATH)
print('✅ Conectado a supply_chain.db')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Conectado a supply_chain.db


## 1️⃣ Leer Bronze — Viajes Diarios

In [ ]:
df = pd.read_sql('SELECT * FROM bronze_viajes_diarios', con)
print(f'📦 Filas Bronze: {len(df):,}')
print(df.dtypes)

📦 Filas Bronze: 1,013
Suborden               object
Do                     object
Estado                 object
Patente                object
Empresa                object
Idruta                 object
Posicionruta           object
Ct                     object
Direccion              object
Localidad              object
Region                 object
Volumen                object
Horainicio             object
Horafin                object
Fechainicioruta        object
Fechapactada           object
Fechaentregareal       object
Fechaestimada          object
Comentarionoentrega    object
Motivonoentrega        object
LPN                    object
LPN_Container          object
Commerce               object
ParentOrder            object
Activa                 object
_ingesta_timestamp     object
_fuente                object
_archivo               object
dtype: object


## 2️⃣ Transformar — silver_viajes

In [ ]:
sv = df.copy()

# ── Renombrar columnas a snake_case limpio ──────────────────────────────────
sv = sv.rename(columns={
    'Suborden'           : 'suborden',
    'Do'                 : 'documento',
    'Estado'             : 'estado',
    'Patente'            : 'patente',
    'Empresa'            : 'empresa',
    'Idruta'             : 'id_ruta',
    'Posicionruta'       : 'posicion_ruta',
    'Ct'                 : 'centro_transporte',
    'Direccion'          : 'direccion',
    'Localidad'          : 'localidad',
    'Region'             : 'region',
    'Volumen'            : 'volumen_raw',
    'Horainicio'         : 'hora_inicio',
    'Horafin'            : 'hora_fin',
    'Fechainicioruta'    : 'fecha_inicio_ruta',
    'Fechapactada'       : 'fecha_pactada',
    'Fechaentregareal'   : 'fecha_entrega_real',
    'Fechaestimada'      : 'fecha_estimada',
    'Comentarionoentrega': 'comentario_no_entrega',
    'Motivonoentrega'    : 'motivo_no_entrega',
    'LPN'                : 'lpn',
    'LPN_Container'      : 'lpn_container',
    'Commerce'           : 'commerce',
    'ParentOrder'        : 'parent_order',
    'Activa'             : 'activa'
})

# ── Eliminar columnas de metadata Bronze ────────────────────────────────────
sv = sv.drop(columns=[c for c in sv.columns if c.startswith('_')])

# ── Tipos: fechas ────────────────────────────────────────────────────────────
fechas = ['fecha_inicio_ruta','fecha_pactada','fecha_entrega_real','fecha_estimada']
for col in fechas:
    sv[col] = pd.to_datetime(sv[col], errors='coerce')

# ── Volumen: limpiar comillas y convertir a float ────────────────────────────
sv['volumen'] = (
    sv['volumen_raw']
    .str.replace('"', '', regex=False)
    .str.strip()
    .astype(float)
)
sv = sv.drop(columns=['volumen_raw'])

# ── Activa: convertir a booleano real ────────────────────────────────────────
sv['activa'] = sv['activa'].map({'True': True, 'False': False, True: True, False: False})

# ── Estado: estandarizar texto ───────────────────────────────────────────────
sv['estado'] = sv['estado'].str.strip().str.title()

# ── Región: limpiar acentos y espacios extra ─────────────────────────────────
sv['region'] = sv['region'].str.strip()

# ── Commerce: estandarizar ───────────────────────────────────────────────────
sv['commerce'] = sv['commerce'].str.strip().str.title()

# ── KPIs de tiempo calculados ────────────────────────────────────────────────
# dias_retraso: diferencia entre entrega real y fecha pactada
sv['dias_retraso'] = (
    sv['fecha_entrega_real'] - sv['fecha_pactada']
).dt.days

# entrega_en_fecha: 1 si entregó en fecha pactada o antes, 0 si tardó
sv['entrega_on_time'] = (sv['dias_retraso'] <= 0).astype(int)

# flag_no_entregado: si el estado NO es 'Terminado'
sv['flag_no_entregado'] = (sv['estado'] != 'Terminado').astype(int)

# ── Timestamp Silver ─────────────────────────────────────────────────────────
sv['_silver_timestamp'] = datetime.now().isoformat()

print(f'✅ silver_viajes procesado: {len(sv):,} filas × {sv.shape[1]} columnas')
print(f'   Nulos en fechas: {sv[fechas].isnull().sum().to_dict()}')
print(f'   Estados únicos: {sv["estado"].value_counts().to_dict()}')
print(f'   Commerce únicos: {sv["commerce"].unique().tolist()}')
sv.head(3)

✅ silver_viajes procesado: 1,013 filas × 29 columnas
   Nulos en fechas: {'fecha_inicio_ruta': 0, 'fecha_pactada': 0, 'fecha_entrega_real': 0, 'fecha_estimada': 0}
   Estados únicos: {'Terminado': 530, 'Pendiente': 292, 'Planificado En Simpliroute': 191}
   Commerce únicos: ['Falabella', 'Sodimac']


,suborden,documento,estado,patente,empresa,id_ruta,posicion_ruta,centro_transporte,direccion,localidad,...,lpn,lpn_container,commerce,parent_order,activa,volumen,dias_retraso,entrega_on_time,flag_no_entregado,_silver_timestamp
0,100000000000000,100000000000000,Terminado,STKY45,LA MEJOR EMPRESA DE TRANSPORTES SPA,14130168,0,HUB XD,Avenida Manuel Rodriguez 694 203,SANTIAGO CENTRO,...,140111000013588362,910000000000907884,Falabella,3227404961,True,0.00048,0,1,0,2026-04-09T01:31:44.351812
1,100000000000000,100000000000000,Terminado,STKY45,LA MEJOR EMPRESA DE TRANSPORTES SPA,14130168,1,HUB XD,Avenida Manuel Rodríguez 694 Depto 413,SANTIAGO CENTRO,...,140111000013605751,910000000000907884,Falabella,3227485971,True,0.00600,0,1,0,2026-04-09T01:31:44.351812
2,100000000000000,100000000000000,Terminado,STKY45,LA MEJOR EMPRESA DE TRANSPORTES SPA,14130168,2,HUB XD,Santo domingo 1669 Departamento 801,SANTIAGO CENTRO,...,140111000013602528,910000000000907884,Falabella,3227471128,True,0.00273,0,1,0,2026-04-09T01:31:44.351812


## 3️⃣ Leer Bronze — Devoluciones y transformar

In [ ]:
dd = pd.read_sql('SELECT * FROM bronze_devoluciones', con)

# Eliminar metadata Bronze
dd = dd.drop(columns=[c for c in dd.columns if c.startswith('_')])

# Tipos: fechas
dd['fecha_viaje']      = pd.to_datetime(dd['fecha_viaje'],      errors='coerce')
dd['fecha_devolucion'] = pd.to_datetime(dd['fecha_devolucion'], errors='coerce')

# Limpiar texto
for col in ['seller','observaciones','estado_recepcion','estado_devolucion','patente']:
    if col in dd.columns:
        dd[col] = dd[col].str.strip().str.title()

# Días entre viaje y devolución
dd['dias_hasta_devolucion'] = (dd['fecha_devolucion'] - dd['fecha_viaje']).dt.days

# Timestamp Silver
dd['_silver_timestamp'] = datetime.now().isoformat()

print(f'✅ silver_devoluciones procesado: {len(dd):,} filas × {dd.shape[1]} columnas')
print(f'   Observaciones únicas: {dd["observaciones"].value_counts().head(5).to_dict()}')
dd.head(3)

✅ silver_devoluciones procesado: 275 filas × 12 columnas
   Observaciones únicas: {'Tienda Errónea': 231, '-': 44}


,nro_recepcion,seller,nro_etiqueta,nro_oc,estado_recepcion,observaciones,patente,fecha_viaje,fecha_devolucion,estado_devolucion,dias_hasta_devolucion,_silver_timestamp
0,CL1773091348393,Sin Información,900600000002966557,-,Recepcionado,Tienda Errónea,Stky72,2026-03-02,2026-03-02,Devuelto,0,2026-04-09T01:32:52.118644
1,CL1773091348393,Sin Información,140121000013560210,-,Recepcionado,Tienda Errónea,Stky72,2026-03-02,2026-03-02,Devuelto,0,2026-04-09T01:32:52.118644
2,CL1773091348393,Sin Información,140121000013613608,-,Recepcionado,Tienda Errónea,Stky72,2026-03-02,2026-03-02,Devuelto,0,2026-04-09T01:32:52.118644


## 4️⃣ Escribir Silver en SQLite

In [ ]:
# Convertir datetimes a string para SQLite
sv_write = sv.copy()
for col in sv_write.select_dtypes(include='datetime64').columns:
    sv_write[col] = sv_write[col].astype(str)

dd_write = dd.copy()
for col in dd_write.select_dtypes(include='datetime64').columns:
    dd_write[col] = dd_write[col].astype(str)

sv_write.to_sql('silver_viajes',       con, if_exists='replace', index=False)
dd_write.to_sql('silver_devoluciones', con, if_exists='replace', index=False)

con.commit()
print('✅ Tablas Silver escritas en SQLite')

✅ Tablas Silver escritas en SQLite


## 5️⃣ Validación Silver

In [ ]:
# Resumen calidad de datos Silver Viajes
sv_check = pd.read_sql('SELECT * FROM silver_viajes', con)

print('=== CALIDAD DATOS — silver_viajes ===')
print(f'Total registros   : {len(sv_check):,}')
print(f'On-Time (%)       : {sv_check["entrega_on_time"].mean()*100:.1f}%')
print(f'No entregados (%) : {sv_check["flag_no_entregado"].mean()*100:.1f}%')
print(f'Volumen total     : {pd.to_numeric(sv_check["volumen"], errors="coerce").sum():.4f}')
print(f'Rutas únicas      : {sv_check["id_ruta"].nunique()}')
print(f'Patentes únicas   : {sv_check["patente"].nunique()}')
print()
print('Distribución por estado:')
print(sv_check['estado'].value_counts())
print()
print('Distribución por Commerce:')
print(sv_check['commerce'].value_counts())

=== CALIDAD DATOS — silver_viajes ===
Total registros   : 1,013
On-Time (%)       : 100.0%
No entregados (%) : 47.7%
Volumen total     : 53.2233
Rutas únicas      : 10
Patentes únicas   : 7

Distribución por estado:
estado
Terminado                     530
Pendiente                     292
Planificado En Simpliroute    191
Name: count, dtype: int64

Distribución por Commerce:
commerce
Falabella    1011
Sodimac         2
Name: count, dtype: int64


In [ ]:
con.close()
print('🔒 Conexión cerrada')


🔒 Conexión cerrada
